# Baseline v4.1 — Iterative DPR (ANN Hard Negatives cho Bi-Encoder)

## Vấn đề với v4
```
v4 MNRL training:
  Batch = [(q1,pos1), (q2,pos2), ...]
  Negative của q1 = pos2, pos3, ... (random, easy)
  → Model học phân biệt chủ đề khác nhau, không học phân biệt passages cùng chủ đề
```

## Giải pháp: ANN Hard Negatives
```
v4.1 training:
  v4_bi.retrieve(q1, top-50) → [pos1, hn1, hn2, hn3, ...]  ← ANN hard negs
  Triplet = (q1, pos1, hn1)  ← explicit hard negative
  MNRL loss = softmax([sim(q,pos), sim(q,hn1), sim(q,in-batch-negs)])
  → Model học phân biệt đúng điều luật vs điều luật gần đúng!
```

| | v4 | v4.1 |
|--|--|--|
| Negative type | Random in-batch | **ANN hard negatives** |
| Khó học | Dễ | **Khó** |
| Kỳ vọng R@1 | 0.5232 | **~0.58-0.65** |

## Cell 0 — Config

In [ ]:
import json, csv, time, random, gc
import numpy as np, faiss, torch
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers import CrossEncoder

ROOT, DATA_DIR = Path("."), Path(".") / "data"
EVAL_DIR = ROOT / "outputs" / "eval"
TMP_DIR  = ROOT / "outputs" / "tmp"
MDL_DIR  = ROOT / "outputs" / "models"

TRAIN_FILE   = DATA_DIR / "train.jsonl"
DEV_FILE     = DATA_DIR / "dev.jsonl"
TRAIN_NEG    = DATA_DIR / "train_with_neg.jsonl"
EVAL_QA_FILE = EVAL_DIR / "eval_qa.jsonl"

# v4 fine-tuned bi-encoder (starting point)
V4_BI_PATH   = MDL_DIR  / "legal_hf_finetuned" / "final"
FAISS_V4     = TMP_DIR  / "faiss_v4.index"
MAP_V4       = TMP_DIR  / "faiss_mapping_v4.jsonl"

# v4.1 output paths
V41_BI_DIR   = MDL_DIR  / "legal_hf_v4_1_iterDPR"
FAISS_V41    = TMP_DIR  / "faiss_v4_1.index"
MAP_V41      = TMP_DIR  / "faiss_mapping_v4_1.jsonl"
RESULT_CSV   = EVAL_DIR / "rerank_metrics_v4_1.csv"

# CE v5 (D_skip14) — tái sử dụng reranker tốt nhất
CE_V5_PATH   = MDL_DIR / "cross_encoder_v5fix" / "saved_model"
if not CE_V5_PATH.exists():
    CE_V5_PATH = MDL_DIR / "cross_encoder_v5fix"

# ── Fine-tuning config ──
FT_EPOCHS    = 5       # 5 epochs như v4
FT_BATCH     = 16      # nhỏ hơn v4 vì triplets tốn VRAM hơn pairs
FT_LR        = 1e-5    # LR nhỏ hơn v4 (2e-5) để không quên kiến thức v4
WARMUP_RATIO = 0.1
MAX_SEQ_LEN  = 256

# ── ANN mining config ──
ANN_TOP_K    = 30      # retrieve top-30 để mine
SKIP_TOP_K   = 3       # bỏ top-3 (có thể là positive hoặc quá khó)
HARD_NEG_PER = 1       # 1 hard negative / query (triplet: q, pos, neg)

# ── Eval config ──
ENCODE_BATCH = 64 if torch.cuda.is_available() else 16
CE_BATCH     = 32
TOP_N_EVAL   = 50
SEED         = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED)
V41_BI_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device      : {DEVICE}")
print(f"Start model : {V4_BI_PATH}")
print(f"FT config   : epochs={FT_EPOCHS}, batch={FT_BATCH}, lr={FT_LR}")
print(f"ANN mining  : top-{ANN_TOP_K}, skip-{SKIP_TOP_K}, {HARD_NEG_PER} neg/q")
print(f"CE v5 path  : {CE_V5_PATH}")
print(f"Target v5   : R@1=0.5418, R@5=0.7245, MRR=0.6307")

## Cell 1 — Utilities

In [ ]:
def load_jsonl(path):
    rows, err = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line=line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except: err+=1
    if err: print(f"  ⚠ {err} errors")
    return rows

def write_jsonl(path, rows):
    with open(path,"w",encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r,ensure_ascii=False)+"\n")

def is_hit(fid, ec, mapping):
    row=mapping[fid]
    for e in ec:
        ci=e.get("chunk_index",-2)
        if ci!=-1 and row["chunk_index"]==ci: return True
        if row["van_ban"]==e.get("van_ban","") and row["dieu"]==e.get("dieu","") and row["khoan"]==e.get("khoan",""): return True
    return False

def is_pos_meta(cand, meta):
    if cand["van_ban"]==meta.get("van_ban","") and cand["van_ban"]!="" \
       and cand["dieu"]==meta.get("dieu","") and cand["khoan"]==meta.get("khoan",""): return True
    ci=meta.get("chunk_index",-2)
    return ci!=-1 and cand["chunk_index"]==ci

def avg(lst): return round(sum(lst)/len(lst),4) if lst else 0.0
print("Utilities ✓")

## Cell 2 — Load v4 Bi-Encoder + FAISS để Mine ANN Hard Negatives

In [ ]:
print("Loading v4 bi-encoder + FAISS...")
v4_bi      = SentenceTransformer(str(V4_BI_PATH), device=DEVICE)
index_v4   = faiss.read_index(str(FAISS_V4))
mapping_v4 = load_jsonl(MAP_V4)
print(f"  ✓ dim={v4_bi.get_sentence_embedding_dimension()} | {index_v4.ntotal} vectors")

## Cell 3 — ANN Hard Negative Mining

Tạo **triplets (query, positive, ann_hard_negative)** cho MNRL:

```
v4_bi.retrieve(query, top-30)
  rank 1-3:   skip (có thể là positive hoặc quá gần positive)
  rank 4-30:  lấy 1 cái không phải positive → ANN hard negative
```

In [ ]:
pos_rows = [r for r in load_jsonl(TRAIN_NEG) if r.get("label")==1]
random.seed(SEED); random.shuffle(pos_rows)
print(f"Positive rows: {len(pos_rows)}")
print(f"ANN mining: top-{ANN_TOP_K}, skip top-{SKIP_TOP_K}, take rank {SKIP_TOP_K+1}-{ANN_TOP_K}")

# Output: 2 loại examples
triplets   = []  # (query, pos, ann_hard_neg) → MNRL hiệu quả nhất
pairs_only = []  # (query, pos) khi không tìm được HN → vẫn học được
stats = {"triplets":0, "pairs":0, "no_hn":0}

for r in tqdm(pos_rows, desc="ANN mining"):
    query = r.get("query","").strip()
    pos_p = r.get("passage","").strip()
    meta  = r.get("meta",{})
    if not query or not pos_p: continue

    # Retrieve top-K với v4 bi-encoder
    q_emb  = v4_bi.encode([query], normalize_embeddings=True,
                           convert_to_numpy=True).astype("float32")
    _, ids = index_v4.search(q_emb, ANN_TOP_K)
    ids    = ids[0].tolist()

    # Lấy ANN hard negative từ rank SKIP_TOP_K+1 trở đi
    ann_hn = None
    for fid in ids[SKIP_TOP_K:]:
        if fid < 0: continue
        cand = mapping_v4[fid]
        if cand["passage"] == pos_p: continue
        if is_pos_meta(cand, meta): continue
        ann_hn = cand["passage"]
        break

    if ann_hn:
        # Triplet: (query, positive, ann_hard_neg)
        triplets.append(InputExample(texts=[query, pos_p, ann_hn]))
        stats["triplets"] += 1
    else:
        # Không tìm được HN → dùng pair thông thường
        pairs_only.append(InputExample(texts=[query, pos_p]))
        stats["pairs"] += 1
        stats["no_hn"] += 1

print(f"\n── Mining Results ──")
print(f"  Triplets (q,pos,ann_hn) : {stats['triplets']}")
print(f"  Pairs only (q,pos)      : {stats['pairs']}")
print(f"  Total training examples : {stats['triplets']+stats['pairs']}")
print(f"  ANN HN coverage         : {stats['triplets']/(stats['triplets']+stats['pairs']):.1%}")

# Kết hợp: triplets trước, pairs sau
all_examples = triplets + pairs_only
random.seed(SEED); random.shuffle(all_examples)
print(f"  Shuffled total          : {len(all_examples)} examples")

## Cell 4 — Fine-tune v4.1 với MNRL + ANN Triplets

> ⏱️ Ước tính: ~30-60 phút (RTX 3050 Ti, 5 epochs)  
> 💡 **MultipleNegativesRankingLoss với triplets** = loss xét cả:
> - In-batch negatives (tất cả positives khác trong batch)
> - Explicit hard negative (ann_hn trong triplet)
> → Model học 2 tầng khó cùng lúc

In [ ]:
# Giải phóng VRAM trước khi fine-tune
del v4_bi; gc.collect()
torch.cuda.empty_cache() if DEVICE=="cuda" else None
print("VRAM cleared ✓")

# Load v4 bi-encoder (starting point của v4.1)
print(f"Loading v4 bi-encoder for fine-tuning: {V4_BI_PATH}")
v41_bi = SentenceTransformer(str(V4_BI_PATH), device=DEVICE)
v41_bi.max_seq_length = MAX_SEQ_LEN
print(f"  Loaded ✓ | dim={v41_bi.get_sentence_embedding_dimension()}")

# DataLoader + Loss
train_dl   = DataLoader(all_examples, shuffle=True, batch_size=FT_BATCH)
train_loss = losses.MultipleNegativesRankingLoss(v41_bi)
# MNRL tự động xử lý cả pairs và triplets:
# - pairs (q, pos): dùng in-batch negatives
# - triplets (q, pos, neg): dùng in-batch + explicit neg

warmup_steps = int(len(train_dl) * FT_EPOCHS * WARMUP_RATIO)
total_steps  = len(train_dl) * FT_EPOCHS
print(f"  Total steps: {total_steps} | Warmup: {warmup_steps}")
print(f"  LR: {FT_LR} (lower than v4's 2e-5 to preserve knowledge)")

print(f"\nFine-tuning v4.1: {FT_EPOCHS} epochs, batch={FT_BATCH}...")
t0 = time.time()
v41_bi.fit(
    train_objectives=[(train_dl, train_loss)],
    epochs=FT_EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": FT_LR},
    use_amp=(DEVICE=="cuda"),
    show_progress_bar=True,
    checkpoint_path=str(V41_BI_DIR),
    checkpoint_save_steps=len(train_dl),
    checkpoint_save_total_limit=2,
)
elapsed = round((time.time()-t0)/60,1)
print(f"Training done in {elapsed} min")

# Save
final_path = V41_BI_DIR / "final"
v41_bi.save(str(final_path))
cfg = {"base":str(V4_BI_PATH), "epochs":FT_EPOCHS, "batch":FT_BATCH,
       "lr":FT_LR, "ann_top_k":ANN_TOP_K, "skip_top_k":SKIP_TOP_K,
       "triplets":stats["triplets"], "pairs":stats["pairs"], "minutes":elapsed}
(V41_BI_DIR / "config.json").write_text(json.dumps(cfg,indent=2,ensure_ascii=False),encoding="utf-8")
print(f"Model saved → {final_path}")

## Cell 5 — Encode Corpus + Build FAISS v4.1

In [ ]:
gc.collect(); torch.cuda.empty_cache() if DEVICE=="cuda" else None

final_path = V41_BI_DIR / "final"
print(f"Loading v4.1 bi-encoder: {final_path}")
v41_bi = SentenceTransformer(str(final_path), device=DEVICE)

# Thu thập corpus
seen = {}
for f in [TRAIN_FILE, DEV_FILE, TRAIN_NEG]:
    for r in load_jsonl(f):
        p = r.get("passage","")
        if p and p not in seen:
            meta=r.get("meta",{})
            seen[p]={"passage":p,"chunk_index":meta.get("chunk_index",-1),
                     "van_ban":meta.get("van_ban",""),"chuong":meta.get("chuong",""),
                     "dieu":meta.get("dieu",""),"khoan":meta.get("khoan",""),"diem":meta.get("diem","")}
corpus = list(seen.values())
texts  = [c["passage"] for c in corpus]
print(f"Corpus: {len(corpus)} passages")

print("Encoding...")
t0 = time.perf_counter()
embs = v41_bi.encode(texts, batch_size=ENCODE_BATCH, show_progress_bar=True,
                     normalize_embeddings=True, convert_to_numpy=True).astype("float32")
print(f"Shape: {embs.shape} | Time: {time.perf_counter()-t0:.1f}s")

index_v41 = faiss.IndexFlatIP(embs.shape[1])
index_v41.add(embs)
faiss.write_index(index_v41, str(FAISS_V41))
mapping_v41 = [{"faiss_id":i,**c} for i,c in enumerate(corpus)]
write_jsonl(MAP_V41, mapping_v41)
print(f"FAISS v4.1 saved ✓ | {FAISS_V41}")

## Cell 6 — Evaluate v4.1 Baseline & v4.1 + CE v5

In [ ]:
gc.collect(); torch.cuda.empty_cache() if DEVICE=="cuda" else None

print(f"Loading CE v5: {CE_V5_PATH}")
ce_v5 = CrossEncoder(str(CE_V5_PATH), max_length=256, device=DEVICE)

eval_qa = load_jsonl(EVAL_QA_FILE)
print(f"Eval QA: {len(eval_qa)} questions")

r_base   = {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]}
r_rerank = {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]}

for item in tqdm(eval_qa, desc="Evaluate v4.1"):
    query=item["query"]; ec=item["expected_citations"]
    q_emb=v41_bi.encode([query],normalize_embeddings=True,
                         convert_to_numpy=True).astype("float32")
    _,ids=index_v41.search(q_emb, TOP_N_EVAL); ids=ids[0].tolist()

    # Baseline v4.1
    for k,key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_base[key].append(1 if any(is_hit(i,ec,mapping_v41) for i in ids[:k] if i>=0) else 0)
    mrr=0.0
    for rank,i in enumerate(ids[:10],1):
        if i>=0 and is_hit(i,ec,mapping_v41): mrr=1.0/rank; break
    r_base["MRR@10"].append(mrr)

    # Rerank với CE v5
    cands  = [(mapping_v41[i]["passage"],i) for i in ids if i>=0]
    rscore = ce_v5.predict([[query,c[0]] for c in cands],batch_size=CE_BATCH) if cands else []
    ranked = sorted(zip(rscore,[c[1] for c in cands]),reverse=True)
    r_ids  = [x[1] for x in ranked]

    for k,key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        r_rerank[key].append(1 if any(is_hit(i,ec,mapping_v41) for i in r_ids[:k]) else 0)
    mrr=0.0
    for rank,i in enumerate(r_ids[:10],1):
        if is_hit(i,ec,mapping_v41): mrr=1.0/rank; break
    r_rerank["MRR@10"].append(mrr)

print("\n── v4.1 Results ──")
print(f"  {'Metric':<10} {'v4 baseline':>13} {'v4.1 base':>12} {'v4.1+CE':>10} {'Δ(v41-v4)':>12}")
print("  "+"-"*60)
prev = {"R@1":0.5232,"R@3":0.6594,"R@5":0.7337,"MRR@10":0.6091}
for k,m in [("R@1","Recall@1"),("R@3","Recall@3"),("R@5","Recall@5"),("MRR@10","MRR@10")]:
    b=avg(r_base[k]); re=avg(r_rerank[k])
    d=b-prev[k]; sign="+" if d>=0 else ""
    print(f"  {m:<10} {prev[k]:>13.4f} {b:>12.4f} {re:>10.4f} {sign}{d:>11.4f}")

## Cell 7 — So sánh tổng hợp & Lưu CSV

In [ ]:
# Kết quả các phiên bản
VERSIONS = {
    "v4_baseline":  {"R@1":0.5232,"R@3":0.6594,"R@5":0.7337,"MRR@10":0.6091},
    "v4+CE(v5)":    {"R@1":0.5418,"R@3":0.6873,"R@5":0.7245,"MRR@10":0.6307},
    "v4.1_base":    {m:avg(r_base[m])   for m in ["R@1","R@3","R@5","MRR@10"]},
    "v4.1+CE(v5)":  {m:avg(r_rerank[m]) for m in ["R@1","R@3","R@5","MRR@10"]},
}

print("\n" + "="*90)
print(f"  {'Metric':<10}", end="")
for vn in VERSIONS: print(f" {vn:>15}", end="")
print()
print("="*90)
for m in ["R@1","R@3","R@5","MRR@10"]:
    print(f"  {m:<10}", end="")
    for vn, vr in VERSIONS.items():
        print(f" {vr[m]:>15.4f}", end="")
    print()
print("="*90)

# Kết luận
v41_base  = VERSIONS["v4.1_base"]
v41_rerank = VERSIONS["v4.1+CE(v5)"]
print(f"\n── Gain of v4.1 baseline vs v4 baseline ──")
for m in ["R@1","R@5","MRR@10"]:
    d = v41_base[m] - VERSIONS["v4_baseline"][m]
    sign="+" if d>=0 else ""
    win="✅" if d>0.005 else ("❌" if d<-0.005 else "=")
    print(f"  {m}: {VERSIONS['v4_baseline'][m]:.4f} → {v41_base[m]:.4f}  ({sign}{d:.4f}) {win}")

# Save
rows=[]
for m in ["R@1","R@3","R@5","MRR@10"]:
    rows.append({"metric":m, **{vn:vr[m] for vn,vr in VERSIONS.items()}})
with open(RESULT_CSV,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=["metric"]+list(VERSIONS.keys()))
    w.writeheader(); w.writerows(rows)
print(f"\nSaved → {RESULT_CSV} ✓")